In [ ]:
import os
import sys
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.isotonic import IsotonicRegression
import lightgbm as lgb
from joblib import Parallel, delayed
sys.path.insert(0, "..")
sys.path.insert(0, ".")
from paths import resolve


In [ ]:
"""
Fast high-accuracy NEM price forecaster.

Method
------
Spike-aware direct multi-horizon LightGBM ensemble with six components
per forecast horizon:

  1. BASE L1 model   – regression_l1 on arcsinh(clip(y, p97)). Clipping the
                       target at the 97th percentile removes extreme-spike
                       noise from the base model's loss, giving it a clean
                       training signal for the majority of intervals.

  2. BASE L2 model   – regression (MSE) on arcsinh(y), uncapped target.
                       Blending L1+L2 (Lago et al. 2021) reduces both MAE
                       and RMSE simultaneously. Per-horizon blend weight α
                       tuned on the validation set.

  3. SPIKE classifier – binary LightGBM estimating P(price > SPIKE_THRESHOLD).
                        Trained with scale_pos_weight to handle class imbalance.

  4. SPIKE regressor  – regression_l1 on arcsinh(y) (full range, no clipping).
                        Spike rows receive 10x sample weight.

  5. SPIKE quantile   – P90 quantile regression on arcsinh(y) for a
                        conservative spike ceiling estimate.

  6. DIP classifier + regressor – mirrors spike components for negative/
                        near-zero prices (y < DIP_THRESHOLD).

Final prediction pipeline per horizon:
  a. Blend L1 + L2 base predictions (α tuned on validation).
  b. Apply spike policy (soft blend / gated uplift / hard gate — kind and
     parameters tuned on validation via _spike_policy_score).
  c. Apply dip policy (same three kinds, tuned independently).
  d. Blend model prediction with lag-naive baseline (α tuned on validation).
  e. Isotonic regression calibration fitted on validation set.

Additionally:
  - Per-horizon target-time features (8 feats: hour/dow sin-cos + regime flags)
  - Cross-features (4 feats: doy cycle, SA-NSW spread, region spike count)
  - Exponential recency weights (newest row ~4.5x oldest)

Config reads TARGET, FEATURE_DATASET, HORIZON_HOURS, OUTPUT_RESOLUTION from
environment variables set by 0_Config/0_variables.ipynb.

Based on Lago et al. (2021) review of state-of-the-art EPF methods and
Ziel & Weron (2018) on spike-aware electricity price forecasting.
"""


In [ ]:
# ── Model configuration ────────────────────────────────────────────────────────
PRICE_TRANSFORM_SCALE = 100.0
TEST_MONTHS           = 12
VALID_MONTHS          = 6
SPIKE_THRESHOLD       = 150.0
_DIP_THRESHOLD        = 0.0
BASE_CLIP_PERCENTILE  = 97.0    # raised 95→97: include more of the spike distribution

_SPIKE_UPWEIGHT   = 10.0  # spike row weight multiplier (raised 5→10)

# Low-price specialist: learn downside tails (negative/near-zero prices).
_DIP_UPWEIGHT  = 7.0   # raised 4→7

# ── Training controls ──────────────────────────────────────────────────────────
# No wall-clock budget — use all available compute for maximum accuracy.
N_JOBS                = -1       # outer parallelism (-1 = all cores)
EARLY_STOPPING_ROUNDS = 75       # reduced 150→75 for faster convergence
SPIKE_ES_ROUNDS       = 50       # reduced 100→50

# ── Model hyperparameters ──────────────────────────────────────────────────────
from parameters import (
    LGBM_BASE_PARAMS,
    LGBM_BASE_L2_PARAMS,
    LGBM_CLF_PARAMS,
    LGBM_SPIKE_PARAMS,
    LGBM_SPIKE_Q_PARAMS,
    LGBM_DIP_CLF_PARAMS,
    LGBM_DIP_PARAMS,
    # _SPIKE_THR_GRID,
    # _SPIKE_POW_GRID,
    # _SPIKE_WMAX_GRID,
    # _SPIKE_GATE_W_GRID,
    # _DIP_THR_GRID,
    # _DIP_POW_GRID,
    # _DIP_WMAX_GRID,
    # _DIP_GATE_W_GRID,
)

MODEL_FILE = Path(resolve(f"4_Model/{os.environ['TARGET'].lower()}_model.joblib"))


In [ ]:
# ── Price transforms ───────────────────────────────────────────────────────────

def _to_asinh(y: np.ndarray) -> np.ndarray:
    return np.arcsinh(y / float(PRICE_TRANSFORM_SCALE))


def _from_asinh(y: np.ndarray) -> np.ndarray:
    return np.sinh(y) * float(PRICE_TRANSFORM_SCALE)


# ── Temporal split ─────────────────────────────────────────────────────────────

def _temporal_split(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cutoff_test  = df.index[-1] - pd.DateOffset(months=TEST_MONTHS)
    cutoff_valid = cutoff_test  - pd.DateOffset(months=VALID_MONTHS)
    train = df[df.index <= cutoff_valid]
    valid = df[(df.index > cutoff_valid) & (df.index <= cutoff_test)]
    test  = df[df.index > cutoff_test]
    return train, valid, test


# ── Naive baseline column ──────────────────────────────────────────────────────

def _horizon_naive_col(h: int) -> str:
    """Same-day naive baseline column.

    Feature data is at 5-min resolution; lag 336 = 28h (closest upstream lag
    to same-time-yesterday at 24h = 288 steps, which is not in the lag set).
    """
    return f"{os.environ['TARGET'].lower()}_price_lag_336"


# ── Spike / dip policy helpers ─────────────────────────────────────────────────

_MIN_SPIKE_TRAIN = 20  # minimum spike rows needed to train spike chain per horizon


def _spike_policy_score(y_true: np.ndarray, y_pred: np.ndarray, spike_threshold: float) -> float:
    mae = float(np.mean(np.abs(y_true - y_pred)))
    spike = y_true > spike_threshold
    dip = y_true < _DIP_THRESHOLD
    if int(spike.sum()) == 0 or int((~spike).sum()) == 0:
        return mae
    spike_mae = float(np.mean(np.abs(y_true[spike] - y_pred[spike])))
    non_mae   = float(np.mean(np.abs(y_true[~spike] - y_pred[~spike])))
    dip_mae   = float(np.mean(np.abs(y_true[dip] - y_pred[dip]))) if int(dip.sum()) > 0 else non_mae
    return 2.2 * spike_mae + 1.3 * dip_mae + 1.0 * non_mae + 0.25 * mae


def _pick_spike_source(
    y_base: np.ndarray,
    y_spike: np.ndarray,
    y_q: np.ndarray,
    src: int,
) -> np.ndarray:
    if src == 1:
        return y_q
    if src == 2:
        return np.maximum(y_spike, y_q).astype(np.float32)
    return y_spike


def _apply_spike_policy(
    y_base: np.ndarray,
    spike_prob: np.ndarray,
    y_spike: np.ndarray,
    y_q: np.ndarray,
    kind: int,
    p1: float,
    p2: float,
    p3: float,
    src: int,
) -> np.ndarray:
    src_pred = _pick_spike_source(y_base, y_spike, y_q, src)
    if kind == 0:
        return ((1.0 - spike_prob) * y_base + spike_prob * src_pred).astype(np.float32)
    if kind == 1:
        return _spike_blend(y_base, spike_prob, src_pred, p1, p2, p3)
    gate = spike_prob >= p1
    uplift = np.maximum(src_pred - y_base, 0.0)
    out = y_base.copy()
    out[gate] = y_base[gate] + float(p2) * uplift[gate]
    return out.astype(np.float32)


def _spike_blend(
    y_base: np.ndarray,
    spike_prob: np.ndarray,
    y_spike: np.ndarray,
    p_thr: float,
    p_pow: float,
    w_max: float,
) -> np.ndarray:
    p_adj = np.clip((spike_prob - p_thr) / (1.0 - p_thr + 1e-6), 0.0, 1.0)
    w     = (np.power(p_adj, p_pow) * w_max).astype(np.float32)
    delta = np.maximum(y_spike - y_base, 0.0)
    return (y_base + w * delta).astype(np.float32)


def _dip_blend(
    y_base: np.ndarray,
    dip_prob: np.ndarray,
    y_dip: np.ndarray,
    p_thr: float,
    p_pow: float,
    w_max: float,
) -> np.ndarray:
    p_adj = np.clip((dip_prob - p_thr) / (1.0 - p_thr + 1e-6), 0.0, 1.0)
    w     = (np.power(p_adj, p_pow) * w_max).astype(np.float32)
    down  = np.maximum(y_base - y_dip, 0.0)
    return (y_base - w * down).astype(np.float32)


def _apply_dip_policy(
    y_base: np.ndarray,
    dip_prob: np.ndarray,
    y_dip: np.ndarray,
    kind: int,
    p1: float,
    p2: float,
    p3: float,
) -> np.ndarray:
    if kind == 0:
        return ((1.0 - dip_prob) * y_base + dip_prob * y_dip).astype(np.float32)
    if kind == 1:
        return _dip_blend(y_base, dip_prob, y_dip, p1, p2, p3)
    gate = dip_prob >= p1
    down = np.maximum(y_base - y_dip, 0.0)
    out = y_base.copy()
    out[gate] = y_base[gate] - float(p2) * down[gate]
    return out.astype(np.float32)


# ── Per-horizon feature builders ───────────────────────────────────────────────

_TARGET_TIME_FEAT_NAMES = [
    "target_hour_sin",
    "target_hour_cos",
    "target_dow_sin",
    "target_dow_cos",
    "target_is_weekend",
    "target_is_peak",
    "target_is_shoulder",
    "target_is_offpeak",
]


def _compute_target_time_feats(idx: pd.DatetimeIndex, h: int) -> np.ndarray:
    """Return (N, 8) float32 array of target-time features for horizon h.

    h is a horizon bucket index (1-based). Each bucket is OUTPUT_RESOLUTION minutes wide.
    """
    _interval = int(os.environ["OUTPUT_RESOLUTION"])
    total_m = idx.hour * 60 + idx.minute + h * _interval
    t_hr = (total_m % 1440) / 60.0
    t_dow = (idx.dayofweek + total_m // 1440) % 7
    return np.column_stack([
        np.sin(2 * np.pi * t_hr / 24.0).astype(np.float32),
        np.cos(2 * np.pi * t_hr / 24.0).astype(np.float32),
        np.sin(2 * np.pi * t_dow / 7.0).astype(np.float32),
        np.cos(2 * np.pi * t_dow / 7.0).astype(np.float32),
        (t_dow >= 5).astype(np.float32),
        ((t_hr >= 17.0) & (t_hr < 21.0)).astype(np.float32),
        ((t_hr >= 7.0) & (t_hr < 17.0)).astype(np.float32),
        ((t_hr < 7.0) | (t_hr >= 21.0)).astype(np.float32),
    ])


In [ ]:
# ── Main training entry point ──────────────────────────────────────────────────

def train_seq2seq(
    X: pd.DataFrame,
    *,
    targets: pd.DataFrame,
    auxiliary: pd.DataFrame,
    force_reselect: bool = False,
) -> dict:
    del force_reselect

    feature_cols = list(X.columns)

    # Horizon is inferred from the target columns (h1, h2, … hN).
    horizon = len([c for c in targets.columns if c.startswith("h") and c[1:].isdigit()])

    train_df, valid_df, _ = _temporal_split(X)

    # Targets and auxiliary sliced to match the same temporal boundaries.
    train_tgt = targets.loc[train_df.index]
    valid_tgt = targets.loc[valid_df.index]
    train_aux = auxiliary.loc[train_df.index]
    valid_aux = auxiliary.loc[valid_df.index]

    X_train  = train_df
    X_valid  = valid_df

    # Cross-features pre-computed in 2_Features_build/9_auxiliary_features.ipynb.
    _CROSS_COLS = ["doy_sin", "doy_cos", "sa_spread_live", "region_spike_score"]
    cross_tr = train_aux[_CROSS_COLS].values.astype(np.float32)
    cross_va = valid_aux[_CROSS_COLS].values.astype(np.float32)

    # Exponential recency weights: newest row ~4.5x oldest (raised from 2x).
    # Stronger recency proved beneficial for NEM markets with regime shifts
    # (Uniejewski et al. 2019: forecasters that up-weight recent observations
    # consistently outperform equal-weight baselines on electricity prices).
    _n_train   = len(train_df)
    _recency_w = np.exp(np.linspace(0.0, 1.5, _n_train)).astype(np.float32)

    # Global p97 clip threshold for base model (avoids per-horizon recomputation).
    _target_lower = os.environ["TARGET"].lower()
    _price_col = f"{_target_lower}_price"
    _all_prices = train_aux[_price_col].values if _price_col in train_aux.columns else (
        train_tgt[[c for c in train_tgt.columns if c.startswith("h") and c[1:].isdigit()]].values.ravel()
    )
    _all_prices = _all_prices[np.isfinite(_all_prices)]
    _clip_thresh = float(np.percentile(_all_prices, BASE_CLIP_PERCENTILE)) if len(_all_prices) > 0 else 300.0

    steps = list(range(1, horizon + 1))
    t0    = time.perf_counter()

    print(
        f"  Fitting {horizon} horizons x 6 models "
        f"(train={len(train_df):,}, valid={len(valid_df):,}, "
        f"clip_p{BASE_CLIP_PERCENTILE:.0f}=${_clip_thresh:.0f}, spike_thr=${SPIKE_THRESHOLD:.0f})",
        flush=True,
    )

    def _fit_one(h: int) -> tuple:
        h_start    = time.perf_counter()
        print(f"    [h={h:02d}] start", flush=True)

        target_col = f"h{h}"
        y_tr_raw   = train_tgt[target_col].values.astype(np.float64)
        y_va_raw   = valid_tgt[target_col].values.astype(np.float64)

        # Build full per-horizon feature matrix.
        h_feat_tr  = _compute_target_time_feats(train_df.index, h)
        h_feat_va  = _compute_target_time_feats(valid_df.index, h)
        X_h_tr = np.concatenate([X_train.values, h_feat_tr, cross_tr], axis=1).astype(np.float32)
        X_h_va = np.concatenate([X_valid.values, h_feat_va, cross_va], axis=1).astype(np.float32)

        # ── 1. Base models ─────────────────────────────────────────────────────
        # Sample weights: recency × moderate spike-target upweighting.
        base_spike_w = (_recency_w * np.where(y_tr_raw > SPIKE_THRESHOLD, 3.0, 1.0)).astype(np.float32)

        # 1a. L1 base (MAE-optimal, clipped target)
        y_tr_base = _to_asinh(np.minimum(y_tr_raw, _clip_thresh)).astype(np.float32)
        y_va_base = _to_asinh(np.minimum(y_va_raw, _clip_thresh)).astype(np.float32)

        base_m = lgb.LGBMRegressor(**LGBM_BASE_PARAMS)
        base_m.fit(
            X_h_tr, y_tr_base,
            sample_weight=base_spike_w,
            eval_set=[(X_h_va, y_va_base)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )

        # 1b. L2 base (MSE-optimal, uncapped target)
        # Blending L1+L2 per Lago et al. (2021) reduces both MAE and RMSE.
        y_tr_full_b = _to_asinh(y_tr_raw).astype(np.float32)
        y_va_full_b = _to_asinh(y_va_raw).astype(np.float32)
        base_l2_m = lgb.LGBMRegressor(**LGBM_BASE_L2_PARAMS)
        base_l2_m.fit(
            X_h_tr, y_tr_full_b,
            sample_weight=base_spike_w,
            eval_set=[(X_h_va, y_va_full_b)],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False)],
        )

        # ── 2. Spike classifier ────────────────────────────────────────────────
        spike_labels_tr = (y_tr_raw > SPIKE_THRESHOLD).astype(np.float32)
        n_spikes_tr     = int(spike_labels_tr.sum())
        spike_clf = None
        spike_reg = None
        spike_qreg = None
        dip_clf = None
        dip_reg = None

        if n_spikes_tr >= _MIN_SPIKE_TRAIN:
            spike_labels_va = (y_va_raw > SPIKE_THRESHOLD).astype(np.float32)
            clf = lgb.LGBMClassifier(**LGBM_CLF_PARAMS)
            clf.fit(
                X_h_tr, spike_labels_tr,
                sample_weight=_recency_w,
                eval_set=[(X_h_va, spike_labels_va)],
                callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
            )
            spike_clf = clf

            # ── 3. Spike regressor (full range, upweighted spike rows) ────────
            y_tr_full = _to_asinh(y_tr_raw).astype(np.float32)
            y_va_full = _to_asinh(y_va_raw).astype(np.float32)
            spike_w   = _recency_w * np.where(spike_labels_tr > 0, _SPIKE_UPWEIGHT, 1.0)

            sreg = lgb.LGBMRegressor(**LGBM_SPIKE_PARAMS)
            sreg.fit(
                X_h_tr, y_tr_full,
                sample_weight=spike_w,
                eval_set=[(X_h_va, y_va_full)],
                callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
            )
            spike_reg = sreg

            qreg = lgb.LGBMRegressor(**LGBM_SPIKE_Q_PARAMS)
            qreg.fit(
                X_h_tr, y_tr_full,
                sample_weight=spike_w,
                eval_set=[(X_h_va, y_va_full)],
                callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
            )
            spike_qreg = qreg

            # ── 4. Dip classifier + regressor (negative/low prices) ─────────
            dip_labels_tr = (y_tr_raw < _DIP_THRESHOLD).astype(np.float32)
            n_dips_tr     = int(dip_labels_tr.sum())
            if n_dips_tr >= _MIN_SPIKE_TRAIN:
                dip_labels_va = (y_va_raw < _DIP_THRESHOLD).astype(np.float32)
                dclf = lgb.LGBMClassifier(**LGBM_DIP_CLF_PARAMS)
                dclf.fit(
                    X_h_tr, dip_labels_tr,
                    sample_weight=_recency_w,
                    eval_set=[(X_h_va, dip_labels_va)],
                    callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
                )
                dip_clf = dclf

                dip_w = _recency_w * np.where(dip_labels_tr > 0, _DIP_UPWEIGHT, 1.0)
                dreg = lgb.LGBMRegressor(**LGBM_DIP_PARAMS)
                dreg.fit(
                    X_h_tr, y_tr_full,
                    sample_weight=dip_w,
                    eval_set=[(X_h_va, y_va_full)],
                    callbacks=[lgb.early_stopping(SPIKE_ES_ROUNDS, verbose=False)],
                )
                dip_reg = dreg

        h_elapsed = time.perf_counter() - h_start
        base_iter = getattr(base_m, "best_iteration_", "?")
        l2_iter   = getattr(base_l2_m, "best_iteration_", "?")
        print(
            f"    [h={h:02d}] done  base_iter={base_iter}/{l2_iter}  "
            f"spike_n={n_spikes_tr}  elapsed={h_elapsed:.1f}s",
            flush=True,
        )
        return base_m, base_l2_m, spike_clf, spike_reg, spike_qreg, dip_clf, dip_reg, h

    os.environ["OMP_NUM_THREADS"] = "1"
    _n_cores = os.cpu_count() or 1
    _parallel_jobs = _n_cores if N_JOBS == -1 else N_JOBS
    print(
        f"  Parallel training: {_parallel_jobs} workers across {len(steps)} horizons "
        f"(N_JOBS={N_JOBS}, logical CPUs={_n_cores})",
        flush=True,
    )
    fitted = Parallel(n_jobs=N_JOBS, prefer="threads")(
        delayed(_fit_one)(h) for h in steps
    )

    elapsed_min = (time.perf_counter() - t0) / 60.0
    print(f"  LGBM total training time: {elapsed_min:.2f} min ({elapsed_min * 60:.0f}s)", flush=True)

    fitted_sorted  = sorted(fitted, key=lambda x: x[7])
    base_models    = [x[0] for x in fitted_sorted]
    base_l2_models = [x[1] for x in fitted_sorted]
    spike_clfs     = [x[2] for x in fitted_sorted]
    spike_regs     = [x[3] for x in fitted_sorted]
    spike_qregs    = [x[4] for x in fitted_sorted]
    dip_clfs       = [x[5] for x in fitted_sorted]
    dip_regs       = [x[6] for x in fitted_sorted]

    # ── Validation tuning – Step A: L1/L2 base blend ───────────────────────────
    # Per-horizon mixing of L1 (median-optimal) and L2 (mean-optimal) bases.
    blend_l2_alphas = np.zeros(horizon, dtype=np.float32)
    _l2_alpha_grid  = np.linspace(0.0, 0.45, 10, dtype=np.float32)
    for _i, _h in enumerate(steps):
        _hf_va  = _compute_target_time_feats(valid_df.index, _h)
        _X_va   = np.concatenate([X_valid.values, _hf_va, cross_va], axis=1).astype(np.float32)
        _y_tv   = valid_tgt[f"h{_h}"].values.astype(np.float32)
        _mask_v = np.isfinite(_y_tv)
        if not np.any(_mask_v):
            continue
        _y_l1 = _from_asinh(base_models[_i].predict(_X_va)).astype(np.float32)
        _y_l2 = _from_asinh(base_l2_models[_i].predict(_X_va)).astype(np.float32)
        _best_a, _best_mae = 0.0, float("inf")
        for _a in _l2_alpha_grid:
            _yc  = ((1.0 - _a) * _y_l1 + _a * _y_l2).astype(np.float32)
            _mae = float(np.mean(np.abs(_y_tv[_mask_v] - _yc[_mask_v])))
            if _mae < _best_mae:
                _best_mae, _best_a = _mae, float(_a)
        blend_l2_alphas[_i] = _best_a
    print(
        f"  L1/L2 blend: mean α={blend_l2_alphas.mean():.3f}  "
        f"min={blend_l2_alphas.min():.3f}  max={blend_l2_alphas.max():.3f}",
        flush=True,
    )

    # ── Validation tuning – Step B: spike/dip policy + naive blend ─────────────
    blend_alphas = np.ones(horizon, dtype=np.float32)
    alpha_grid   = np.linspace(0.0, 1.0, 41, dtype=np.float32)  # finer grid (was 21)
    spike_kind   = np.zeros(horizon, dtype=np.int8)     # 0=soft, 1=uplift, 2=hard-gate
    spike_src    = np.zeros(horizon, dtype=np.int8)     # 0=sr, 1=sq, 2=max(sr,sq)
    spike_p1     = np.full(horizon, 0.20, dtype=np.float32)
    spike_p2     = np.full(horizon, 2.0, dtype=np.float32)
    spike_p3     = np.full(horizon, 0.75, dtype=np.float32)
    dip_kind     = np.zeros(horizon, dtype=np.int8)     # 0=soft, 1=down-gate, 2=hard-gate
    dip_p1       = np.full(horizon, 0.15, dtype=np.float32)
    dip_p2       = np.full(horizon, 1.0, dtype=np.float32)
    dip_p3       = np.full(horizon, 0.60, dtype=np.float32)

    for i, h in enumerate(steps):
        naive_col = _horizon_naive_col(h)
        if naive_col not in valid_aux.columns:
            naive_col = f"{_target_lower}_price_lag_48"
        if naive_col not in valid_aux.columns:
            continue

        naive_h_val = valid_aux[naive_col].values.astype(np.float32)
        y_true      = valid_tgt[f"h{h}"].values.astype(np.float32)
        mask        = np.isfinite(y_true) & np.isfinite(naive_h_val)
        if not np.any(mask):
            continue

        h_feat_va  = _compute_target_time_feats(valid_df.index, h)
        X_h_va = np.concatenate([X_valid.values, h_feat_va, cross_va], axis=1).astype(np.float32)

        _l1_val    = _from_asinh(base_models[i].predict(X_h_va)).astype(np.float32)
        _l2_val    = _from_asinh(base_l2_models[i].predict(X_h_va)).astype(np.float32)
        y_base_val = ((1.0 - blend_l2_alphas[i]) * _l1_val + blend_l2_alphas[i] * _l2_val).astype(np.float32)
        y_model    = y_base_val
        if spike_clfs[i] is not None:
            sp     = spike_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
            sr     = _from_asinh(spike_regs[i].predict(X_h_va)).astype(np.float32) if spike_regs[i] else y_base_val
            sq     = _from_asinh(spike_qregs[i].predict(X_h_va)).astype(np.float32) if spike_qregs[i] else sr

            y_tune = y_true[mask]
            b_tune = y_base_val[mask]
            s_tune = sp[mask]
            r_tune = sr[mask]
            q_tune = sq[mask]

            # Reference (old behavior): soft blend with spike reg only.
            y_ref = _apply_spike_policy(b_tune, s_tune, r_tune, q_tune, 0, 0.0, 0.0, 0.0, 0)
            ref_spike = y_tune > SPIKE_THRESHOLD
            if int((~ref_spike).sum()) > 0:
                ref_non_mae = float(np.mean(np.abs(y_tune[~ref_spike] - y_ref[~ref_spike])))
            else:
                ref_non_mae = float(np.mean(np.abs(y_tune - y_ref)))

            best_sc = _spike_policy_score(y_tune, y_ref, SPIKE_THRESHOLD)
            best_k, best_src = 0, 0
            best_p1, best_p2, best_p3 = 0.0, 0.0, 0.0

            def _accept_candidate(y_candidate: np.ndarray) -> tuple[bool, float]:
                m = np.abs(y_tune - y_candidate)
                if int((~ref_spike).sum()) > 0:
                    cand_non = float(np.mean(m[~ref_spike]))
                else:
                    cand_non = float(np.mean(m))
                if cand_non > ref_non_mae + 0.20:
                    return False, float("inf")
                return True, _spike_policy_score(y_tune, y_candidate, SPIKE_THRESHOLD)

            # Policy 0: soft blend (src variants)
            for src in (0, 1, 2):
                y_try = _apply_spike_policy(b_tune, s_tune, r_tune, q_tune, 0, 0.0, 0.0, 0.0, src)
                ok, sc = _accept_candidate(y_try)
                if ok and sc < best_sc:
                    best_sc = sc
                    best_k, best_src = 0, src
                    best_p1, best_p2, best_p3 = 0.0, 0.0, 0.0

            # Policy 1: gated uplift (threshold/power/wmax)
            for src in (0, 1, 2):
                for thr in _SPIKE_THR_GRID:
                    for pwr in _SPIKE_POW_GRID:
                        for wmx in _SPIKE_WMAX_GRID:
                            y_try = _apply_spike_policy(
                                b_tune, s_tune, r_tune, q_tune,
                                1, float(thr), float(pwr), float(wmx), src,
                            )
                            ok, sc = _accept_candidate(y_try)
                            if ok and sc < best_sc:
                                best_sc = sc
                                best_k, best_src = 1, src
                                best_p1, best_p2, best_p3 = float(thr), float(pwr), float(wmx)

            # Policy 2: hard gate uplift (threshold, weight)
            for src in (0, 1, 2):
                for thr in _SPIKE_THR_GRID:
                    for gw in _SPIKE_GATE_W_GRID:
                        y_try = _apply_spike_policy(
                            b_tune, s_tune, r_tune, q_tune,
                            2, float(thr), float(gw), 0.0, src,
                        )
                        ok, sc = _accept_candidate(y_try)
                        if ok and sc < best_sc:
                            best_sc = sc
                            best_k, best_src = 2, src
                            best_p1, best_p2, best_p3 = float(thr), float(gw), 0.0

            spike_kind[i] = best_k
            spike_src[i]  = best_src
            spike_p1[i]   = best_p1
            spike_p2[i]   = best_p2
            spike_p3[i]   = best_p3

            y_model = _apply_spike_policy(
                y_base_val,
                sp,
                sr,
                sq,
                int(best_k),
                float(best_p1),
                float(best_p2),
                float(best_p3),
                int(best_src),
            )

        # Dip policy tuning after spike policy so downside corrections are additive.
        if dip_clfs[i] is not None and dip_regs[i] is not None:
            dp = dip_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
            dr = _from_asinh(dip_regs[i].predict(X_h_va)).astype(np.float32)

            y_tune = y_true[mask]
            b_tune = y_model[mask]
            p_tune = dp[mask]
            d_tune = dr[mask]
            ref_mid = (y_tune >= _DIP_THRESHOLD) & (y_tune <= SPIKE_THRESHOLD)
            if int(ref_mid.sum()) > 0:
                ref_mid_mae = float(np.mean(np.abs(y_tune[ref_mid] - b_tune[ref_mid])))
            else:
                ref_mid_mae = float(np.mean(np.abs(y_tune - b_tune)))

            best_d_sc = _spike_policy_score(y_tune, b_tune, SPIKE_THRESHOLD)
            best_dk = 0
            best_dp1, best_dp2, best_dp3 = 0.0, 0.0, 0.0

            def _accept_dip(y_candidate: np.ndarray) -> tuple[bool, float]:
                if int(ref_mid.sum()) > 0:
                    mid_mae = float(np.mean(np.abs(y_tune[ref_mid] - y_candidate[ref_mid])))
                else:
                    mid_mae = float(np.mean(np.abs(y_tune - y_candidate)))
                if mid_mae > ref_mid_mae + 0.15:
                    return False, float("inf")
                return True, _spike_policy_score(y_tune, y_candidate, SPIKE_THRESHOLD)

            y_try = _apply_dip_policy(b_tune, p_tune, d_tune, 0, 0.0, 0.0, 0.0)
            ok, sc = _accept_dip(y_try)
            if ok and sc < best_d_sc:
                best_d_sc = sc
                best_dk = 0
                best_dp1, best_dp2, best_dp3 = 0.0, 0.0, 0.0

            for thr in _DIP_THR_GRID:
                for pwr in _DIP_POW_GRID:
                    for wmx in _DIP_WMAX_GRID:
                        y_try = _apply_dip_policy(
                            b_tune,
                            p_tune,
                            d_tune,
                            1,
                            float(thr),
                            float(pwr),
                            float(wmx),
                        )
                        ok, sc = _accept_dip(y_try)
                        if ok and sc < best_d_sc:
                            best_d_sc = sc
                            best_dk = 1
                            best_dp1, best_dp2, best_dp3 = float(thr), float(pwr), float(wmx)

            for thr in _DIP_THR_GRID:
                for gw in _DIP_GATE_W_GRID:
                    y_try = _apply_dip_policy(
                        b_tune,
                        p_tune,
                        d_tune,
                        2,
                        float(thr),
                        float(gw),
                        0.0,
                    )
                    ok, sc = _accept_dip(y_try)
                    if ok and sc < best_d_sc:
                        best_d_sc = sc
                        best_dk = 2
                        best_dp1, best_dp2, best_dp3 = float(thr), float(gw), 0.0

            dip_kind[i] = best_dk
            dip_p1[i]   = best_dp1
            dip_p2[i]   = best_dp2
            dip_p3[i]   = best_dp3

            y_model = _apply_dip_policy(
                y_model,
                dp,
                dr,
                int(best_dk),
                float(best_dp1),
                float(best_dp2),
                float(best_dp3),
            )

        y_t, y_m, y_n = y_true[mask], y_model[mask], naive_h_val[mask]
        best_a, best_mae = 1.0, float("inf")
        for a in alpha_grid:
            mae = float(np.mean(np.abs(y_t - (a * y_m + (1.0 - a) * y_n))))
            if mae < best_mae:
                best_mae, best_a = mae, float(a)
        blend_alphas[i] = best_a

    # ── Isotonic calibration ───────────────────────────────────────────────────
    calibrators: list[IsotonicRegression | None] = [None] * horizon

    for i, h in enumerate(steps):
        naive_col = _horizon_naive_col(h)
        if naive_col not in valid_aux.columns:
            naive_col = f"{_target_lower}_price_lag_48"
        if naive_col not in valid_aux.columns:
            continue

        h_feat_va  = _compute_target_time_feats(valid_df.index, h)
        X_h_va = np.concatenate([X_valid.values, h_feat_va, cross_va], axis=1).astype(np.float32)

        _l1_cal    = _from_asinh(base_models[i].predict(X_h_va)).astype(np.float32)
        _l2_cal    = _from_asinh(base_l2_models[i].predict(X_h_va)).astype(np.float32)
        y_base_val = ((1.0 - blend_l2_alphas[i]) * _l1_cal + blend_l2_alphas[i] * _l2_cal).astype(np.float32)
        if spike_clfs[i] is not None:
            sp     = spike_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
            sr     = _from_asinh(spike_regs[i].predict(X_h_va)).astype(np.float32) if spike_regs[i] else y_base_val
            sq     = _from_asinh(spike_qregs[i].predict(X_h_va)).astype(np.float32) if spike_qregs[i] else sr
            y_model = _apply_spike_policy(
                y_base_val,
                sp,
                sr,
                sq,
                int(spike_kind[i]),
                float(spike_p1[i]),
                float(spike_p2[i]),
                float(spike_p3[i]),
                int(spike_src[i]),
            )
        else:
            y_model = y_base_val

        if i < len(dip_clfs) and dip_clfs[i] is not None and i < len(dip_regs) and dip_regs[i] is not None:
            dp = dip_clfs[i].predict_proba(X_h_va)[:, 1].astype(np.float32)
            dr = _from_asinh(dip_regs[i].predict(X_h_va)).astype(np.float32)
            kind = int(dip_kind[i]) if i < len(dip_kind) else 0
            p1   = float(dip_p1[i]) if i < len(dip_p1) else 0.15
            p2   = float(dip_p2[i]) if i < len(dip_p2) else 1.0
            p3   = float(dip_p3[i]) if i < len(dip_p3) else 0.60
            y_model = _apply_dip_policy(y_model, dp, dr, kind, p1, p2, p3)

        y_naive = valid_aux[naive_col].values.astype(np.float32)
        a       = float(blend_alphas[i])
        y_blend = a * y_model + (1.0 - a) * y_naive
        y_true  = valid_tgt[f"h{h}"].values.astype(np.float32)
        mask    = np.isfinite(y_true) & np.isfinite(y_blend)
        if int(mask.sum()) < 500:
            continue

        iso = IsotonicRegression(out_of_bounds="clip")
        iso.fit(y_blend[mask], y_true[mask])
        calibrators[i] = iso

    return {
        "horizon":         horizon,
        "base_models":     base_models,
        "base_l2_models":  base_l2_models,
        "blend_l2_alphas": blend_l2_alphas,
        "spike_clfs":      spike_clfs,
        "spike_regs":      spike_regs,
        "spike_qregs":     spike_qregs,
        "dip_clfs":        dip_clfs,
        "dip_regs":        dip_regs,
        "feature_cols":    feature_cols,
        "blend_alphas":    blend_alphas,
        "spike_kind":   spike_kind,
        "spike_src":    spike_src,
        "spike_p1":     spike_p1,
        "spike_p2":     spike_p2,
        "spike_p3":     spike_p3,
        "dip_kind":     dip_kind,
        "dip_p1":       dip_p1,
        "dip_p2":       dip_p2,
        "dip_p3":       dip_p3,
        "calibrators":  calibrators,
        "clip_thresh":  _clip_thresh,
    }


In [ ]:
_feat_stem = Path(os.environ["FEATURE_DATASET"]).stem

feature_path = resolve(f"2_Features_build/Feature_data/{_feat_stem}.parquet")
targets_path = resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_targets_agg_{_feat_stem}.parquet")
selected_path = resolve(f"3_Features_select/Selected_features/{os.environ['TARGET']}_selected_features_{_feat_stem}.parquet")
auxiliary_path = resolve(f"2_Features_build/Feature_data/{os.environ['TARGET']}_auxiliary_{_feat_stem}.parquet")

features = pd.read_parquet(feature_path)
targets = pd.read_parquet(targets_path)
auxiliary = pd.read_parquet(auxiliary_path)

selected = pd.read_parquet(selected_path)
selected_features = [f for f in selected.index[selected.any(axis=1)] if f in features.columns]


In [ ]:
model = train_seq2seq(
    features[selected_features],
    auxiliary=auxiliary,
    targets=targets,
)
